# CounterFeint - Compare runs

Aggregates baseline + post-training results into a single set of plots + tables for the README and slide deck.

## What it expects to find

| Path | Source |
|---|---|
| `baseline_outputs/<model_tag>/baseline_results.json` | `baseline_eval.ipynb` |
| `outputs/<run_tag>/eval_summary.json`                | `official_hf_training.ipynb` |
| `outputs/<run_tag>/log_history.json`                 | `official_hf_training.ipynb` |
| `outputs/<run_tag>/training_config.json`             | `official_hf_training.ipynb` |

You can run multiple training jobs back-to-back with different `RUN_TAG` values (e.g. `qwen3-0.6b-r16`, `qwen3-0.6b-r32`, `qwen2.5-1.5b-r16`) and this notebook will pull them all into one comparison.

## Outputs

- `comparison_outputs/before_after_grader.png` - bar chart of mean grader_score per task, baseline vs each trained run
- `comparison_outputs/training_curves.png` - reward + loss curves overlaid for all runs
- `comparison_outputs/comparison_table.csv` - one row per (run, task) for the README

These are the figures you drop into your README and slide deck.

In [ ]:
import json
from pathlib import Path
from typing import Any, Dict, List

import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "counterfeint" / "server").exists()),
    Path.cwd(),
)
print(f"REPO_ROOT = {REPO_ROOT}")

BASELINE_DIR = REPO_ROOT / "baseline_outputs"
TRAINED_DIR  = REPO_ROOT / "outputs"
OUT_DIR      = REPO_ROOT / "comparison_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"baseline_dir = {BASELINE_DIR}  (exists={BASELINE_DIR.exists()})")
print(f"trained_dir  = {TRAINED_DIR}   (exists={TRAINED_DIR.exists()})")

In [ ]:
def _episode_score(r: Dict[str, Any]) -> float:
    return float(r.get("grader_score") or 0.0)


def per_task_means(rows: List[Dict[str, Any]]) -> Dict[str, float]:
    by_task: Dict[str, List[float]] = {}
    for r in rows:
        by_task.setdefault(r["task_id"], []).append(_episode_score(r))
    out = {t: (sum(v) / len(v) if v else 0.0) for t, v in by_task.items()}
    if rows:
        out["overall"] = sum(_episode_score(r) for r in rows) / len(rows)
    else:
        out["overall"] = 0.0
    return out


def collect_baselines(baseline_dir: Path) -> List[Dict[str, Any]]:
    out = []
    if not baseline_dir.exists():
        return out
    for sub in sorted(p for p in baseline_dir.iterdir() if p.is_dir()):
        f = sub / "baseline_results.json"
        if not f.exists():
            continue
        data = json.loads(f.read_text())
        rows = data.get("rows", [])
        out.append({
            "kind": "baseline",
            "tag": data.get("model_tag", sub.name),
            "model": data.get("model_repo", "?"),
            "means": per_task_means(rows),
            "rows": rows,
        })
    return out


def collect_trained(trained_dir: Path) -> List[Dict[str, Any]]:
    out = []
    if not trained_dir.exists():
        return out
    for sub in sorted(p for p in trained_dir.iterdir() if p.is_dir()):
        f = sub / "eval_summary.json"
        if not f.exists():
            continue
        data = json.loads(f.read_text())
        # eval_summary.json from official_hf_training.ipynb has shape:
        # { "summary": {...}, "before": [episode_dict, ...], "after": [...] }
        before_rows = data.get("before") or []
        after_rows  = data.get("after")  or []
        # base_model lives in training_config.json next to it.
        base_model = "?"
        cfg_path = sub / "training_config.json"
        if cfg_path.exists():
            try:
                cfg = json.loads(cfg_path.read_text())
                base_model = cfg.get("base_model") or cfg.get("BASE_MODEL", "?")
            except Exception:
                pass
        log_json = sub / "log_history.json"
        out.append({
            "kind": "trained",
            "tag": sub.name,
            "model": base_model,
            "before_means": per_task_means(before_rows),
            "after_means":  per_task_means(after_rows),
            "before_rows": before_rows,
            "after_rows":  after_rows,
            "log_history_json": str(log_json) if log_json.exists() else None,
        })
    return out


baselines = collect_baselines(BASELINE_DIR)
trained = collect_trained(TRAINED_DIR)
print(f"Found {len(baselines)} baseline run(s) and {len(trained)} trained run(s).")
for b in baselines:
    print(f"  baseline: {b['tag']:<20} -> overall={b['means']['overall']:.3f}")
for t in trained:
    print(
        f"  trained : {t['tag']:<20} -> "
        f"before={t['before_means']['overall']:.3f}  "
        f"after={t['after_means']['overall']:.3f}  "
        f"delta={(t['after_means']['overall']-t['before_means']['overall']):+.3f}"
    )

## Section 2 - Per-task mean grader_score (baseline vs trained)

The headline plot for the README. Bars are mean grader_score per task; one group per run.

In [ ]:
labels = []
overall = []
t1, t2, t3 = [], [], []
colors = []

for b in baselines:
    labels.append(f"base:{b['tag']}")
    overall.append(b["means"]["overall"])
    t1.append(b["means"].get("task_1", 0.0))
    t2.append(b["means"].get("task_2", 0.0))
    t3.append(b["means"].get("task_3", 0.0))
    colors.append("#888888")

for t in trained:
    labels.append(f"trained:{t['tag']}")
    overall.append(t["after_means"]["overall"])
    t1.append(t["after_means"].get("task_1", 0.0))
    t2.append(t["after_means"].get("task_2", 0.0))
    t3.append(t["after_means"].get("task_3", 0.0))
    colors.append("#4C72B0")

if labels:
    width = 0.20
    x = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(max(8, 0.9 * len(labels) + 4), 5.5))
    ax.bar(x - 1.5 * width, t1, width, label="task_1", color="#4C72B0")
    ax.bar(x - 0.5 * width, t2, width, label="task_2", color="#55A868")
    ax.bar(x + 0.5 * width, t3, width, label="task_3", color="#C44E52")
    ax.bar(x + 1.5 * width, overall, width, label="overall", color="#8172B2")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_ylabel("Mean grader_score")
    ax.set_title("Per-task grader_score: baselines vs trained")
    ax.set_ylim(0, 1.0)
    ax.axhline(0.45, ls="--", color="gray", lw=1, label="trainable bar (0.45)")
    ax.axhline(0.80, ls="--", color="green", lw=1, label="strong bar (0.80)")
    ax.legend(ncol=3, fontsize=9, loc="upper right")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    out_path = OUT_DIR / "before_after_grader.png"
    fig.savefig(out_path, dpi=140, bbox_inches="tight")
    plt.show()
    print(f"Wrote {out_path}")
else:
    print("No runs found. Run baseline_eval.ipynb and/or official_hf_training.ipynb first.")

## Section 3 - Training curves overlaid

Reads `log_history.json` from each trained run and overlays reward, loss, and KL curves.

In [ ]:
def load_log_history(path: str):
    raw = json.loads(Path(path).read_text())
    rows = []
    for entry in raw:
        step = entry.get("step")
        if step is None:
            continue
        rows.append({
            "step":   int(step),
            "loss":   entry.get("loss"),
            "reward": entry.get("reward"),
            "kl":     entry.get("kl"),
        })
    return rows


curves = []
for t in trained:
    if not t["log_history_json"]:
        continue
    try:
        rows = load_log_history(t["log_history_json"])
    except Exception as exc:
        print(f"  [skip] {t['tag']}: cannot read log_history.json ({exc})")
        continue
    if not rows:
        continue
    curves.append({"tag": t["tag"], "rows": rows})

if not curves:
    print("No log_history.json files found yet. Run official_hf_training.ipynb first.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for c in curves:
        steps   = [r["step"]   for r in c["rows"]]
        losses  = [r["loss"]   for r in c["rows"]]
        rewards = [r["reward"] for r in c["rows"]]
        kls     = [r["kl"]     for r in c["rows"]]
        s_l = [s for s, v in zip(steps, losses)  if v is not None]
        v_l = [v for v in losses  if v is not None]
        s_r = [s for s, v in zip(steps, rewards) if v is not None]
        v_r = [v for v in rewards if v is not None]
        s_k = [s for s, v in zip(steps, kls)     if v is not None]
        v_k = [v for v in kls     if v is not None]
        if v_l: axes[0].plot(s_l, v_l, label=c["tag"], lw=1.5)
        if v_r: axes[1].plot(s_r, v_r, label=c["tag"], lw=1.5)
        if v_k: axes[2].plot(s_k, v_k, label=c["tag"], lw=1.5)
    axes[0].set_title("GRPO loss");        axes[0].set_xlabel("step"); axes[0].set_ylabel("loss")
    axes[1].set_title("GRPO mean reward"); axes[1].set_xlabel("step"); axes[1].set_ylabel("reward")
    axes[2].set_title("GRPO KL divergence"); axes[2].set_xlabel("step"); axes[2].set_ylabel("kl")
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend(fontsize=9)
    fig.tight_layout()
    out_path = OUT_DIR / "training_curves.png"
    fig.savefig(out_path, dpi=140, bbox_inches="tight")
    plt.show()
    print(f"Wrote {out_path}")

## Section 4 - CSV table for the README

In [ ]:
import csv

table_rows = []
for b in baselines:
    for task in ("task_1", "task_2", "task_3", "overall"):
        table_rows.append({
            "kind": "baseline",
            "run_tag": b["tag"],
            "model": b["model"],
            "task": task,
            "score": round(b["means"].get(task, 0.0), 4),
            "delta_vs_baseline": "",
        })

baseline_lookup = {(b["tag"], task): b["means"].get(task, 0.0) for b in baselines for task in ("task_1", "task_2", "task_3", "overall")}

for t in trained:
    base_match = next((b for b in baselines if b["model"] == t["model"]), None)
    for task in ("task_1", "task_2", "task_3", "overall"):
        score = t["after_means"].get(task, 0.0)
        delta = ""
        if base_match:
            delta = round(score - base_match["means"].get(task, 0.0), 4)
        table_rows.append({
            "kind": "trained",
            "run_tag": t["tag"],
            "model": t["model"],
            "task": task,
            "score": round(score, 4),
            "delta_vs_baseline": delta,
        })

if table_rows:
    csv_path = OUT_DIR / "comparison_table.csv"
    with open(csv_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(table_rows[0].keys()))
        w.writeheader()
        w.writerows(table_rows)
    print(f"Wrote {csv_path}")
    print(f"\n{'kind':<10} {'run_tag':<22} {'task':<10} {'score':>8} {'delta':>8}")
    for r in table_rows:
        print(f"{r['kind']:<10} {r['run_tag']:<22} {r['task']:<10} {r['score']:>8} {str(r['delta_vs_baseline']):>8}")
else:
    print("Nothing to tabulate yet.")